# SRA Dynamic Synaptic Hot-Swap Experiment
## Merging an independently trained Spanish translation model into a running French/German translation model.

This notebook demonstrates how SRA's routing mechanism reacts when external synapses are forcefully injected. We verify the necessity of a 'Shared Trunk' (shared base representation) to prevent routing collapse (Representation Divergence).

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import random
import numpy as np
import sys
sys.path.append('../src')
from sra_language_models import MoESRALanguageModel

# 再現性のためのシード固定
torch.manual_seed(42)
random.seed(42)

### 1. Dataset and Utility Definition
Define a simple vocabulary and translation dataset for 4 languages.

In [ ]:
PARALLEL_DATA = [
    {"eng": "hello", "fra": "bonjour", "deu": "hallo", "spa": "hola"},
    {"eng": "apple", "fra": "pomme", "deu": "apfel", "spa": "manzana"},
    {"eng": "cat", "fra": "chat", "deu": "katze", "spa": "gato"},
    {"eng": "dog", "fra": "chien", "deu": "hund", "spa": "perro"},
    {"eng": "water", "fra": "eau", "deu": "wasser", "spa": "agua"},
    {"eng": "friend", "fra": "ami", "deu": "freund", "spa": "amigo"},
    {"eng": "thank you", "fra": "merci", "deu": "danke", "spa": "gracias"},
    {"eng": "goodbye", "fra": "au revoir", "deu": "auf wiedersehen", "spa": "adios"},
    {"eng": "house", "fra": "maison", "deu": "haus", "spa": "casa"},
    {"eng": "car", "fra": "voiture", "deu": "auto", "spa": "coche"}
]

vocab = ["[PAD]", "[ENG]", "[FRA]", "[DEU]", "[SPA]", "[SEP]", "[EOS]"]
for pair in PARALLEL_DATA:
    for word in pair.values():
        if word not in vocab:
            vocab.append(word)

word2idx = {w: i for i, w in enumerate(vocab)}
idx2word = {i: w for i, w in enumerate(vocab)}
VOCAB_SIZE = len(vocab)

def encode(lang_tag, src_word, tgt_word=None):
    prompt = [word2idx[lang_tag], word2idx[src_word], word2idx["[SEP]"], word2idx["[ENG]"]]
    if tgt_word:
        target = [word2idx[tgt_word], word2idx["[EOS]"]]
        return prompt + target
    return prompt

print(f"Vocab Size: {VOCAB_SIZE}")


In [ ]:
def get_batch(langs, batch_size=16):
    x = torch.zeros((batch_size, 6), dtype=torch.long)
    y = torch.full((batch_size, 6), -100, dtype=torch.long)
    
    for i in range(batch_size):
        lang = random.choice(langs)
        lang_tag = f"[{lang.upper()}]"
        pair = random.choice(PARALLEL_DATA)
        seq = encode(lang_tag, pair[lang], pair["eng"])
        x[i, :5] = torch.tensor(seq[:5])
        y[i, 3:5] = torch.tensor(seq[4:6])
        
    return x, y

def train_model(model, langs, steps=400):
    optimizer = torch.optim.AdamW(model.parameters(), lr=2e-3)
    model.train()
    for step in range(steps):
        x, y = get_batch(langs)
        logits, router_logits = model(x)
        loss = F.cross_entropy(logits.view(-1, VOCAB_SIZE), y.view(-1), ignore_index=-100)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    print(f"Trained on {langs} - Final Loss: {loss.item():.4f}")

def test_model(model, langs_to_test):
    model.eval()
    with torch.no_grad():
        for lang in langs_to_test:
            lang_tag = f"[{lang.upper()}]"
            pair = PARALLEL_DATA[0] # Test with 'hello'
            src_word = pair[lang]
            expected = pair["eng"]
            
            x = torch.tensor([encode(lang_tag, src_word)]).long()
            logits, router_logits = model(x)
            
            pred_idx = logits[0, 3].argmax().item()
            pred_word = idx2word[pred_idx]
            
            success = pred_word == expected
            status = "✅" if success else "❌"
            print(f"{status} {lang.upper()} -> ENG: '{src_word}' -> '{pred_word}' (Expected: '{expected}')")
            
            if router_logits:
                probs = F.softmax(router_logits[0][0, 3], dim=-1)
                top_synapse = probs.argmax().item()
                print(f"   [Routed to Synapse {top_synapse}] probs: {[round(p, 3) for p in probs.tolist()]}")

## 3. Model 1 と Model 2 の独立学習

In [ ]:
DIM = 64
model1 = MoESRALanguageModel(vocab_size=VOCAB_SIZE, dim=DIM, layers=1, num_synapses=2, k=1, syn_hidden=128)
model2 = MoESRALanguageModel(vocab_size=VOCAB_SIZE, dim=DIM, layers=1, num_synapses=1, k=1, syn_hidden=128)

print("Training Model 1 (fra, deu)...")
train_model(model1, ["fra", "deu"], steps=600)

print("\nTraining Model 2 (spa)...")
train_model(model2, ["spa"], steps=600)

print("\n--- Model 1 Test ---")
test_model(model1, ["fra", "deu", "spa"])

## 4. ホットスワップ機能の実装（ADD）
SRAのベクトル化されたエキスパート（CausalMoESRABlock）に対して、テンソルの次元方向（dim=0）にパラメータを結合（cat）するだけで、動的にシナプスを追加できます。

In [ ]:
def hotswap_add_synapse(target_model, source_model):
    print("\n🔄 稼働中の Model 1 にスペイン語シナプスをホットスワップ注入（テンソル結合）します...")
    layer = target_model.blocks[0]
    src_layer = source_model.blocks[0]
    
    # Parameter tensor concatenation
    with torch.no_grad():
        layer.w1.data = torch.cat([layer.w1.data, src_layer.w1.data[0:1]], dim=0)
        layer.b1.data = torch.cat([layer.b1.data, src_layer.b1.data[0:1]], dim=0)
        layer.w2.data = torch.cat([layer.w2.data, src_layer.w2.data[0:1]], dim=0)
        layer.b2.data = torch.cat([layer.b2.data, src_layer.b2.data[0:1]], dim=0)
        layer.state.data = torch.cat([layer.state.data, src_layer.state.data[0:1]], dim=0)
        
        # Router Key concatenation (dim=0 is the output dimension for nn.Linear weight)
        old_key = layer.router.synapse_emb.data
        src_key = src_layer.router.synapse_emb.data
        new_key = torch.cat([old_key, src_key[0:1]], dim=0)
        
        layer.router.synapse_emb = nn.Parameter(new_key)
        layer.router.num_synapses += 1
        layer.num_synapses += 1
        
    print("✅ ホットスワップ完了！")

hotswap_add_synapse(model1, model2)

print("\n--- Model 1 Test (After Hot-Swap ADD) ---")
test_model(model1, ["fra", "deu", "spa"])

## 5. ホットスワップ機能の実装（DELETE / Lesion）
Model 1 から「フランス語」のシナプスパラメータを切り落とします。

In [ ]:
def hotswap_delete_synapse(target_model, idx_to_remove):
    print(f"\n✂️ Model 1 から シナプス {idx_to_remove} (フランス語) を削除します...")
    layer = target_model.blocks[0]
    
    with torch.no_grad():
        keep_indices = [i for i in range(layer.num_synapses) if i != idx_to_remove]
        
        layer.w1.data = layer.w1.data[keep_indices]
        layer.b1.data = layer.b1.data[keep_indices]
        layer.w2.data = layer.w2.data[keep_indices]
        layer.b2.data = layer.b2.data[keep_indices]
        layer.state.data = layer.state.data[keep_indices]
        
        new_key = layer.router.synapse_emb.data[keep_indices]
        layer.router.synapse_emb = nn.Parameter(new_key)
        
        layer.router.num_synapses -= 1
        layer.num_synapses -= 1
        
    print("✅ 削除完了！")

# フランス語のルーティング先を調べる
model1.eval()
with torch.no_grad():
    x = torch.tensor([encode("[FRA]", "bonjour")]).long()
    _, router_logits = model1(x)
    fra_synapse_idx = router_logits[0][0, 3].argmax().item()
    
hotswap_delete_synapse(model1, fra_synapse_idx)

print("\n--- Model 1 Test (After Hot-Swap DELETE) ---")
test_model(model1, ["fra", "deu", "spa"])